## Cell 1 — Install dependencies

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q --upgrade bitsandbytes
!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    trl==0.10.1 \
    accelerate==0.34.0 \
    datasets==2.21.0
!pip install -q --upgrade wandb

print("Dependencies installed")

## Cell 2 — Verify GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

## Cell 3 — Imports

In [ ]:
import torch
import wandb
import logging
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)
from trl import DPOTrainer, DPOConfig
from kaggle_secrets import UserSecretsClient

torch.manual_seed(42)
np.random.seed(42)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
print("All imports successful")

## Cell 4 — Login

In [ ]:
secrets = UserSecretsClient()

wandb_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

hf_token = secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=hf_token)

print("W&B and HuggingFace login successful")

## Cell 5 — Config

In [ ]:
HF_USERNAME = "YOUR_HF_USERNAME"  # your actual username here in Kaggle

config = {
    "model": {
        "sft_model_path": f"{HF_USERNAME}/alignforge-phi3-sft",
        "max_seq_length": 1024,
    },
    "quantization": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_use_double_quant": True,
    },
    "lora": {
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": [
            "q_proj", "k_proj", "v_proj",
            "o_proj", "gate_proj", "up_proj", "down_proj"
        ],
    },
    "dpo": {
        "beta": 0.1,
        "loss_type": "sigmoid",
        "max_prompt_length": 512,
        "max_length": 1024,
    },
    "training": {
        "output_dir": "/kaggle/working/outputs/dpo",
        "num_train_epochs": 1,
        "per_device_train_batch_size": 1,
        "per_device_eval_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "gradient_checkpointing": True,
        "learning_rate": 5e-5,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.1,
        "fp16": True,
        "logging_steps": 10,
        "eval_steps": 200,
        "save_steps": 200,
        "save_total_limit": 2,
        "load_best_model_at_end": True,
        "report_to": "wandb",
        "run_name": "alignforge-dpo",
    },
    "dataset": {
        "name": "Anthropic/hh-rlhf",
        "max_samples": 5000,
    }
}

print("Config loaded")

## Cell 6 — VRAM utility

In [ ]:
def get_vram():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free = total - reserved
    print(f"Allocated: {allocated:.2f}GB | Reserved: {reserved:.2f}GB | Free: {free:.2f}GB | Total: {total:.2f}GB")

print("Before model load:")
get_vram()

## Cell 7 — Load SFT model as policy

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

logger.info("Loading tokenizer from base model...")
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = "right"

logger.info("Loading policy model (SFT model) in 4-bit...")
policy_model = AutoModelForCausalLM.from_pretrained(
    config["model"]["sft_model_path"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
policy_model.config.use_cache = False

print("After policy model load:")
get_vram()
print("Policy model loaded")

## Cell 8 — Apply LoRA to policy model

In [ ]:
policy_model = prepare_model_for_kbit_training(policy_model)

lora_config = LoraConfig(
    r=config["lora"]["r"],
    lora_alpha=config["lora"]["lora_alpha"],
    lora_dropout=config["lora"]["lora_dropout"],
    target_modules=config["lora"]["target_modules"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

policy_model = get_peft_model(policy_model, lora_config)
policy_model.print_trainable_parameters()

print("After LoRA:")
get_vram()

## Cell 9 — Load reference model

In [ ]:
logger.info("Loading reference model (frozen SFT model)...")
ref_model = AutoModelForCausalLM.from_pretrained(
    config["model"]["sft_model_path"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
ref_model.config.use_cache = False
ref_model.eval()

for param in ref_model.parameters():
    param.requires_grad = False

print("After reference model load:")
get_vram()
print("Reference model loaded and frozen")

## Cell 10 — Prepare DPO dataset

In [ ]:
def extract_prompt_and_responses(sample):
    def parse(text):
        last_assistant = text.rfind("\n\nAssistant:")
        if last_assistant == -1:
            return None, None
        prompt = text[:last_assistant].strip()
        response = text[last_assistant + len("\n\nAssistant:"):].strip()
        return prompt, response

    chosen_prompt, chosen_response = parse(sample["chosen"])
    rejected_prompt, rejected_response = parse(sample["rejected"])

    if not chosen_prompt or not chosen_response or not rejected_response:
        return {
            "prompt": None,
            "chosen": None,
            "rejected": None,
        }

    prompt_formatted = f"<|user|>\n{chosen_prompt}<|end|>\n<|assistant|>\n"
    chosen_formatted = f"<|user|>\n{chosen_prompt}<|end|>\n<|assistant|>\n{chosen_response}<|end|>"
    rejected_formatted = f"<|user|>\n{chosen_prompt}<|end|>\n<|assistant|>\n{rejected_response}<|end|>"

    return {
        "prompt": prompt_formatted,
        "chosen": chosen_formatted,
        "rejected": rejected_formatted,
    }

logger.info("Loading hh-rlhf dataset...")
raw_dataset = load_dataset(
    config["dataset"]["name"],
    split="train"
)

if config["dataset"]["max_samples"]:
    raw_dataset = raw_dataset.select(range(config["dataset"]["max_samples"]))

logger.info("Formatting dataset for DPO...")
dpo_dataset = raw_dataset.map(
    extract_prompt_and_responses,
    remove_columns=raw_dataset.column_names,
    desc="Formatting for DPO"
)

dpo_dataset = dpo_dataset.filter(
    lambda x: x["prompt"] is not None and x["chosen"] is not None and x["rejected"] is not None,
    desc="Filtering empty samples"
)

split = dpo_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print("\nSample prompt:", train_dataset[0]["prompt"][:200])
print("\nSample chosen:", train_dataset[0]["chosen"][200:400])
print("\nSample rejected:", train_dataset[0]["rejected"][200:400])

## Cell 11 — Build and run DPO training

In [ ]:
dpo_config = DPOConfig(
    output_dir=config["training"]["output_dir"],
    num_train_epochs=config["training"]["num_train_epochs"],
    per_device_train_batch_size=config["training"]["per_device_train_batch_size"],
    per_device_eval_batch_size=config["training"]["per_device_eval_batch_size"],
    gradient_accumulation_steps=config["training"]["gradient_accumulation_steps"],
    gradient_checkpointing=config["training"]["gradient_checkpointing"],
    learning_rate=config["training"]["learning_rate"],
    lr_scheduler_type=config["training"]["lr_scheduler_type"],
    warmup_ratio=config["training"]["warmup_ratio"],
    fp16=config["training"]["fp16"],
    logging_steps=config["training"]["logging_steps"],
    eval_strategy="steps",
    eval_steps=config["training"]["eval_steps"],
    save_strategy="steps",
    save_steps=config["training"]["save_steps"],
    save_total_limit=config["training"]["save_total_limit"],
    load_best_model_at_end=config["training"]["load_best_model_at_end"],
    report_to=config["training"]["report_to"],
    run_name=config["training"]["run_name"],
    beta=config["dpo"]["beta"],
    loss_type=config["dpo"]["loss_type"],
    max_prompt_length=config["dpo"]["max_prompt_length"],
    max_length=config["dpo"]["max_length"],
    remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print("DPO Trainer built")
print("Starting DPO training...")
train_result = trainer.train()
print(f"DPO training complete. Loss: {train_result.training_loss:.4f}")

## Cell 12 — Save and push DPO model

In [ ]:
from peft import PeftModel

adapter_path = "/kaggle/working/outputs/dpo/adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"DPO adapter saved to {adapter_path}")

logger.info("Merging DPO LoRA weights...")
base_model = AutoModelForCausalLM.from_pretrained(
    config["model"]["sft_model_path"],
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
merged_dpo = PeftModel.from_pretrained(base_model, adapter_path)
merged_dpo = merged_dpo.merge_and_unload()

merged_path = "/kaggle/working/outputs/dpo/merged"
merged_dpo.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)
print(f"Merged DPO model saved to {merged_path}")

merged_dpo.push_to_hub(f"{HF_USERNAME}/alignforge-phi3-dpo")
tokenizer.push_to_hub(f"{HF_USERNAME}/alignforge-phi3-dpo")
print(f"Pushed to: huggingface.co/{HF_USERNAME}/alignforge-phi3-dpo")